# 🎬 AutoDub Studio — One-Click Colab Deployment

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syedj7895-cell/ai-video-dubber-AutoDub-Studio-Automatic-Dubbing-Engine-ElevenLabs-Quality-Open-Source-/blob/main/Colab_Runner.ipynb)

ElevenLabs-quality **open-source automatic dubbing** — Demucs v4 · Pyannote 3.1 ·
SenseVoice-Small · CosyVoice 3.0 zero-shot cloning — wrapped in an iPhone-15
frosted-glass UI.

**How to use**
1. ▸ *Runtime* ▸ *Change runtime type* ▸ **T4 GPU** ▸ Save
2. ▸ *Runtime* ▸ **Run all**
3. When the last cell finishes, click the printed **`https://….gradio.live`** link
4. In Tab 1: upload your **audio/video master + Original SRT + Translated SRT**, hit 🔍
5. Tab 2: 🧬 matches speakers & emotions (paste a HuggingFace token in Tab 1 ▸ Advanced first — see [token guide](https://github.com/syedj7895-cell/ai-video-dubber-AutoDub-Studio-Automatic-Dubbing-Engine-ElevenLabs-Quality-Open-Source-#-one-time-hugging-face-setup-required-for-step-3))
6. Tab 3: 🚀 renders the dubbed master mix / video

Ready to use — no build step required.

In [ ]:
# ── 1 ▸ Runtime sanity check ──────────────────────────────────────────
import torch
if torch.cuda.is_available():
    print(f"✅ GPU ready: {torch.cuda.get_device_name(0)} "
          f"({torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GB)")
else:
    print("⚠ No GPU detected — the app still runs, but SLOWLY.")
    print("  Fix: Runtime ▸ Change runtime type ▸ T4 GPU ▸ Save, then re-run.")

In [ ]:
# ── 2 ▸ Clone the project ─────────────────────────────────────────────
REPO_URL = ("https://github.com/syedj7895-cell/"
            "ai-video-dubber-AutoDub-Studio-Automatic-Dubbing-Engine-"
            "ElevenLabs-Quality-Open-Source-.git")
!git clone {REPO_URL} ai-video-dubber
%cd ai-video-dubber
!ls

In [ ]:
# ── realign gradio-client to gradio's EXACT pin (Colab ships stale 1.3.0) ──
import importlib.metadata as md, subprocess, sys
try:
    g = md.version("gradio")
    want = next((r.split("==")[1].strip() for r in md.requires("gradio")
                 if r.startswith("gradio-client")), None)
    have = md.version("gradio_client")
    if want and have != want:
        print(f"gradio {g}: gradio-client {have} -> {want}")
        subprocess.check_call([sys.executable, "-m", "pip", "install",
                               "-q", f"gradio-client=={want}"])
    else:
        print(f"gradio {g} + gradio-client {have} already aligned")
except Exception as e:
    print("client realignment skipped:", e)

In [ ]:
# ── CosyVoice source-tree registration (repo has NO setup.py) ──
import os, pathlib, site, subprocess, sys
CV = pathlib.Path("/content/CosyVoice")
assert CV.is_dir() and (CV / "cosyvoice").is_dir(), \
    "Clone it first:  git clone --recursive https://github.com/FunAudioLLM/CosyVoice /content/CosyVoice"
for p in (str(CV), str(CV / "third_party" / "Matcha-TTS")):
    if p not in sys.path:
        sys.path.insert(0, p)
# persist for every future runtime process (incl. Gradio subprocesses)
pth = pathlib.Path(site.getsitepackages()[0]) / "autodub_cosyvoice.pth"
pth.write_text("\n".join([str(CV), str(CV / "third_party" / "Matcha-TTS")]), encoding="utf-8")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "-r", str(CV / "requirements.txt")])
import cosyvoice  # sanity check
print("✅ CosyVoice importable (source-tree mode · no pip install .)")

In [ ]:
# ── 5 ▸ 🚀 LAUNCH — click the https://….gradio.live link printed below ─
!python app.py